In [1]:
import requests, hashlib, time, os, json
from pathlib import Path
from datetime import datetime
from math import log10
import pandas as pd
import numpy as np
from dotenv import load_dotenv

# load_dotenv()
# API_KEY    = os.getenv('PODCASTINDEX_API_KEY')
# API_SECRET = os.getenv('PODCASTINDEX_API_SECRET')
# if not API_KEY or not API_SECRET:
#     raise ValueError('Add PODCASTINDEX_API_KEY and PODCASTINDEX_API_SECRET to your .env file.')

# BASE_URL   = 'https://api.podcastindex.org/api/1.0'
OUTPUT_DIR = Path('../data')
OUTPUT_DIR.mkdir(exist_ok=True)

This is the function clean description. I removed fuzzy matching and adding HTML tag removal

In [2]:
import re
import string
import spacy

nlp = spacy.load('en_core_web_sm', disable=['ner', 'parser'])
DEFAULT_STOPWORDS = nlp.Defaults.stop_words | {"podcast", "episode"}

def clean_description(text, stopwords=DEFAULT_STOPWORDS, do_lower=True):
    """
    Preprocess text by:
    - Removing HTML tags
    - Lowercasing (optional)
    - Removing punctuation
    - Lemmatization
    - Removing stopwords
    """
    if not isinstance(text, str):
        return ''

    # NOTE: moved HTML tag removal to a separate function to handle it before lemmatization
    # # Drop HTML tags like <p>, </div>, etc.
    # text = re.sub(r'<[^>]+>', ' ', text)

    if do_lower:
        text = text.lower()

    text = re.sub(f'[{re.escape(string.punctuation)}]', ' ', text)
    doc = nlp(text)
    words = [token.lemma_ for token in doc if token.is_alpha and token.text not in stopwords]

    result = ' '.join(words)
    result = re.sub(r'\s+', ' ', result).strip()
    return result

trying a new way to clean the html tags :(

In [4]:
import html
from html.parser import HTMLParser

class _HTMLStripper(HTMLParser):
    def __init__(self):
        super().__init__()
        self._parts = []
    def handle_data(self, data):
        self._parts.append(data)
    def get_text(self):
        return ' '.join(self._parts)

def clean_html(raw):
    if not raw or not isinstance(raw, str): return ''
    decoded = html.unescape(raw)
    stripper = _HTMLStripper()
    stripper.feed(decoded)
    return ' '.join(stripper.get_text().split())

def remove_certain_words(text, words_to_remove):
    if not isinstance(text, str):
        return ''
    pattern = r'\b(' + '|'.join(map(re.escape, words_to_remove)) + r')\b'
    return re.sub(pattern, '', text, flags=re.IGNORECASE)
 
df_podcasts = pd.read_csv(OUTPUT_DIR / 'podcasts_cleaned2.csv')
df_episodes = pd.read_csv(OUTPUT_DIR / 'episodes_cleaned2.csv')

df_podcasts['clean_description_html'] = df_podcasts['description'].fillna('').apply(clean_html)
df_episodes['clean_description_html'] = df_episodes['description'].fillna('').apply(clean_html)

# then apply clean_description to the cleaned HTML text
df_podcasts['clean_description_html'] = df_podcasts['clean_description_html'].apply(clean_description)
df_episodes['clean_description_html'] = df_episodes['clean_description_html'].apply(clean_description)

# just remove "https"
df_podcasts['clean_description_html'] = df_podcasts['clean_description_html'].apply(lambda x: remove_certain_words(x, ['https']))
df_episodes['clean_description_html'] = df_episodes['clean_description_html'].apply(lambda x: remove_certain_words(x, ['https']))



PermissionError: [Errno 13] Permission denied: '..\\data\\podcasts_cleanedHTML_NEW.csv'

In [6]:
df_podcasts.to_csv(OUTPUT_DIR / 'podcasts_cleanedHTML_NEW2.csv', index=False)
df_episodes.to_csv(OUTPUT_DIR / 'episodes_cleanedHTML_NEW2.csv', index=False)
print('HTML cleaned from descriptions.')
print('Saved cleaned files as podcasts_cleanedHTML_NEW2.csv and episodes_cleanedHTML_NEW2.csv')

HTML cleaned from descriptions.
Saved cleaned files as podcasts_cleanedHTML_NEW2.csv and episodes_cleanedHTML_NEW2.csv


- use podcasts_cleaned2 and episodes_cleaned2
- the ids of podcasts i will delete
ids = [893264
770043
1012966
651031
806626
369640
878280
321024
160055
75761
200302
134118
298899
277707
114271
28841
1212629
1218701
1317354
1116937
365303
551519
323319
50924
50727
50925
50738
84759
2502409
2738040
1372516
743742
339409
309633
280808
346335
154186
350534
322275
161453
512528
42962
271599
176845
183737
88679
162542
132840
92548
11232
181500
105287
22665
4359565
176543
339287
109826
16898
216250
784025
633023
800975
249596
634718
368589
797983
219378
]

In [18]:
import pandas as pd

INPUT_PODCASTS = OUTPUT_DIR / "podcasts_cleaned2.csv"
INPUT_EPISODES = OUTPUT_DIR / "episodes_cleaned2.csv"
OUTPUT_PODCASTS = OUTPUT_DIR / "podcasts_cleaned2_filtered.csv"
OUTPUT_EPISODES = OUTPUT_DIR / "episodes_cleaned2_filtered.csv"

# Paste podcast IDs to remove here.
podcast_ids_to_remove = {
    "893264", "770043", "1012966", "651031", "806626", "369640", "878280", "321024",
    "160055", "75761", "200302", "134118", "298899", "277707", "114271", "28841",
    "1212629", "1218701", "1317354", "1116937", "365303", "551519", "323319", "50924",
    "50727", "50925", "50738", "84759", "2502409", "2738040", "1372516", "743742",
    "339409", "309633", "280808", "346335", "154186", "350534", "322275", "161453",
    "512528", "42962", "271599", "176845", "183737", "88679", "162542", "132840",
    "92548", "11232", "181500", "105287", "22665", "4359565", "176543", "339287",
    "109826", "16898", "216250", "784025", "633023", "800975", "249596", "634718",
    "368589", "797983", "219378"
}

def find_column(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None

df_podcasts = pd.read_csv(INPUT_PODCASTS)
df_episodes = pd.read_csv(INPUT_EPISODES)

podcast_id_col = find_column(df_podcasts, ["id", "podcast_id", "podcastId", "show_id"])
episode_fk_col = find_column(df_episodes, ["podcast_id", "podcastId", "show_id", "podcastid", "feed_id"])

if podcast_id_col is None:
    raise ValueError(
        f"Could not find podcast ID column in podcasts file. Columns: {list(df_podcasts.columns)}"
    )
if episode_fk_col is None:
    raise ValueError(
        f"Could not find podcast foreign-key column in episodes file. Columns: {list(df_episodes.columns)}"
    )

before_podcasts = len(df_podcasts)
before_episodes = len(df_episodes)

podcast_ids_series = df_podcasts[podcast_id_col].astype(str).str.strip()
episode_fk_series = df_episodes[episode_fk_col].astype(str).str.strip()

df_podcasts_filtered = df_podcasts[~podcast_ids_series.isin(podcast_ids_to_remove)].copy()
df_episodes_filtered = df_episodes[~episode_fk_series.isin(podcast_ids_to_remove)].copy()

after_podcasts = len(df_podcasts_filtered)
after_episodes = len(df_episodes_filtered)

df_podcasts_filtered.to_csv(OUTPUT_PODCASTS, index=False)
df_episodes_filtered.to_csv(OUTPUT_EPISODES, index=False)

remaining_bad_links = (
    df_episodes_filtered[episode_fk_col].astype(str).str.strip().isin(podcast_ids_to_remove).sum()
)

print(f"Podcast ID column: {podcast_id_col}")
print(f"Episode FK column: {episode_fk_col}")
print(f"Podcasts: {before_podcasts} -> {after_podcasts} (removed {before_podcasts - after_podcasts})")
print(f"Episodes: {before_episodes} -> {after_episodes} (removed {before_episodes - after_episodes})")
print(f"Episodes still linked to removed podcasts: {remaining_bad_links}")
print(f"Saved: {OUTPUT_PODCASTS}")
print(f"Saved: {OUTPUT_EPISODES}")

Podcast ID column: id
Episode FK column: podcast_id
Podcasts: 750 -> 683 (removed 67)
Episodes: 13930 -> 12779 (removed 1151)
Episodes still linked to removed podcasts: 0
Saved: ..\data\podcasts_cleaned2_filtered.csv
Saved: ..\data\episodes_cleaned2_filtered.csv


Process episodes.

In [ ]:
# INPUT_PATH = OUTPUT_DIR / 'episodes_cleaned2.csv'
# OUTPUT_PATH = OUTPUT_DIR / 'episodes_cleanedHTML2.csv'
# CHUNKSIZE = 50
# first = True
# for chunk in pd.read_csv(INPUT_PATH, chunksize=CHUNKSIZE):
#     chunk['clean_description'] = chunk['description'].apply(clean_description)
#     chunk.to_csv(OUTPUT_PATH, mode='w' if first else 'a', index=False, header=first)
#     first = False
#     print(f"Processed {len(chunk)} episode rows...")

# print(f"Saved cleaned episode data to {OUTPUT_PATH}")

Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 e

Other person runs this: ESTELLA run this one - NOT USING THIS PART

In [ ]:
# INPUT_PATH = OUTPUT_DIR / 'episodes_cleaned2.csv'
# OUTPUT_PATH1 =  OUTPUT_DIR / 'episodes_cleanedpart1.csv'
# OUTPUT_PATH2 =  OUTPUT_DIR / 'episodes_cleanedpart2.csv'
# first = False
# for chunk in pd.read_csv(INPUT_PATH, chunksize=CHUNKSIZE):
#     chunk['clean_description'] = chunk['description'].apply(clean_description)
#     chunk.to_csv(OUTPUT_PATH2, mode='w' if first else 'a', index=False, header=first)
#     first = False
#     print(f"Processed {len(chunk)} episode rows...")

# print(f"Saved cleaned episode data to {OUTPUT_PATH2}")

In [ ]:
#Combine back into one csv called episodes_cleaned2HTML.csv
# OUTPUT_PATH1 =  OUTPUT_DIR / 'episodes_cleanedHTML1.csv'
# OUTPUT_PATH2 =  OUTPUT_DIR / 'episodes_cleanedHTML2.csv'
# df1 = pd.read_csv(OUTPUT_PATH1)
# df2 = pd.read_csv(OUTPUT_PATH2)
# combined_df = pd.concat([df1, df2], ignore_index=True)
# combined_df.to_csv(OUTPUT_DIR / 'episodes_cleanedHTML2.csv', index=False)
# print(f"Combined cleaned episode data saved to {OUTPUT_DIR / 'episodes_cleanedHTML2.csv'}")

Open the csv files and check that most tags are removed beore continuing please please please

Get embeddings now with the cleaned html csv  files. First for podcasts

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

df_podcasts = pd.read_csv(OUTPUT_DIR / 'podcasts_cleaned2_filtered.csv')
df_episodes = pd.read_csv(OUTPUT_DIR / 'episodes_cleaned2_filtered.csv')

# Build text strings: for podcasts, combine name + description + categories; for episodes, combine episode name + description.
def podcast_text(r):
    cats = str(r.get('categories', '')).replace('|', ' ')
    return f"{r['name']} {cats} {r.get('clean_description_html', '')}"

def ep_text(r):
    return f"{r['episode_name']} {r.get('clean_description_html', '')}"

show_texts = df_podcasts.apply(podcast_text, axis=1).tolist()
ep_texts   = df_episodes.apply(ep_text,     axis=1).tolist()
show_ids   = df_podcasts['id'].astype(str).tolist()
ep_ids     = df_episodes['id'].astype(str).tolist()

# Podcast SVD embeddings
N_COMPONENTS = 100  # number of SVD dimensions 100-300 for now cuz um thats what i learned

tfidf_shows = TfidfVectorizer(max_features=10000, stop_words='english')
tfidf_matrix_shows = tfidf_shows.fit_transform(show_texts) # sparse (n_podcasts, 10000)
print(f'Podcast TF-IDF matrix: {tfidf_matrix_shows.shape}')

svd_shows = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)
show_embs = svd_shows.fit_transform(tfidf_matrix_shows)  # dense (n_podcasts, 100)
print(f'Explained variance: {svd_shows.explained_variance_ratio_.sum():.2%}')

np.save(OUTPUT_DIR / 'podcasts_embeddings.npy', show_embs.astype(np.float32))
Path(OUTPUT_DIR / 'podcasts_embeddings_ids.txt').write_text('\n'.join(show_ids))
print(f'Podcast embeddings: shape={show_embs.shape} | {(OUTPUT_DIR/"podcasts_embeddings.npy").stat().st_size/1024:.0f} KB')

Podcast TF-IDF matrix: (750, 8985)
Explained variance: 36.41%
Podcast embeddings: shape=(750, 100) | 293 KB


Create episode embeddings

In [14]:
# Episode SVD embeddings:
def episode_embeddings(file_num):
    # df_episodes = pd.read_csv(OUTPUT_DIR / f'episodes_cleanedHTML{file_num}.csv')
    tfidf_eps = TfidfVectorizer(max_features=10000, stop_words='english')
    tfidf_matrix_eps = tfidf_eps.fit_transform(ep_texts)
    print(f'Episode TF-IDF matrix: {tfidf_matrix_eps.shape}')

    svd_eps = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)
    ep_embs = svd_eps.fit_transform(tfidf_matrix_eps)

    np.save(OUTPUT_DIR / f'episodes_embeddings.npy', ep_embs.astype(np.float32))
    Path(OUTPUT_DIR / f'episodes_embeddings_ids.txt').write_text('\n'.join(ep_ids))
    print(f'Episode embeddings:  shape={ep_embs.shape} | {(OUTPUT_DIR/f"episodes_embeddings.npy").stat().st_size/1024:.0f} KB')

    # Save vectorizers + SVD models (needed at query time!) 
    import pickle
    # for podcasts saving
    with open(OUTPUT_DIR / f'svd_podcasts.pkl', 'wb') as f:
        pickle.dump({'tfidf': tfidf_shows, 'svd': svd_shows}, f)
        
    # for episodes saving
    with open(OUTPUT_DIR / f'svd_episodes.pkl', 'wb') as f:
        pickle.dump({'tfidf': tfidf_eps, 'svd': svd_eps}, f)
    print(f'Saved svd_podcasts.pkl and svd_episodes.pkl')
    

episode_embeddings(2)

Episode TF-IDF matrix: (13930, 10000)
Episode embeddings:  shape=(13930, 100) | 5442 KB
Saved svd_podcasts.pkl and svd_episodes.pkl


Now get the mixed embeddings

In [ ]:
# Build mixed podcast embeddings in the SAME latent space as podcast SVD
import pickle
from collections import defaultdict
from sklearn.preprocessing import normalize

# Load embeddings and IDs already created above
show_embs = np.load(OUTPUT_DIR / 'podcasts_embeddings.npy')  # (n_podcasts, n_components)
show_ids = [line.strip() for line in (OUTPUT_DIR / 'podcasts_embeddings_ids.txt').read_text().splitlines()]

# Load cleaned data
df_podcasts = pd.read_csv(OUTPUT_DIR / 'podcasts_cleaned2_filtered.csv')
df_episodes = pd.read_csv(OUTPUT_DIR / 'episodes_cleaned2_filtered.csv')

# Aggregate episode text by podcast
podcast_to_episode_text = defaultdict(list)
for row in df_episodes.itertuples(index=False):
    podcast_to_episode_text[str(row.podcast_id)].append(f"{row.episode_name} {row.clean_description}")

# Create one aggregated episode document per podcast in podcast order
episode_docs_by_show = []
for sid in show_ids:
    episode_docs_by_show.append(" ".join(podcast_to_episode_text.get(str(sid), [])))

# Project aggregated episode docs into the EXISTING podcast TF-IDF + SVD space
episode_tfidf_in_show_space = tfidf_shows.transform(episode_docs_by_show)
episode_embs_in_show_space = svd_shows.transform(episode_tfidf_in_show_space)

# L2-normalize both components before mixing
show_embs_norm = normalize(show_embs, norm='l2')
episode_embs_norm = normalize(episode_embs_in_show_space, norm='l2')

# Weighted mix in one shared space
alpha_episode = 0.1
alpha_show = 0.9
new_show_embs = alpha_episode * episode_embs_norm + alpha_show * show_embs_norm
new_show_embs = normalize(new_show_embs, norm='l2')

# Save mixed embeddings + IDs
np.save(OUTPUT_DIR / 'embeddings_mixed.npy', new_show_embs.astype(np.float32))
Path(OUTPUT_DIR / 'podcasts_embeddings_ids.txt').write_text('\n'.join(show_ids))
print(f"Saved mixed podcast embeddings: shape={new_show_embs.shape} | {(OUTPUT_DIR/'embeddings_mixed.npy').stat().st_size/1024:.0f} KB")

# Save query-time model for mixed retrieval (same vectorizer/SVD used to embed queries)
with open(OUTPUT_DIR / 'svd_mixed.pkl', 'wb') as f:
    pickle.dump({'tfidf': tfidf_shows, 'svd': svd_shows, 'mix_weights': {'episode': alpha_episode, 'show': alpha_show}}, f)

print('Saved svd_mixed.pkl for query-time projection in shared space.')

Saved mixed podcast embeddings: shape=(750, 100) | 293 KB
Saved svd_mixed.pkl for query-time projection in shared space.
